In [1]:
import os
import openreview
from tqdm import tqdm
from datetime import datetime
import pandas as pd
from time import sleep

# Test OpenReview API

In [2]:
client = openreview.api.OpenReviewClient(
    baseurl='https://api2.openreview.net',
)

In [3]:
name = "Lovkush Agarwal"
profiles = client.search_profiles(term=name)

In [4]:
print(len(profiles))
print(profiles[0])

1
{'active': True,
 'content': {'emails': ['****@gmail.com'],
             'emailsConfirmed': ['****@gmail.com'],
             'gender': 'Male',
             'gscholar': 'https://scholar.google.com/citations?view_op=list_works&hl=en&user=nm-XFVoAAAAJ',
             'history': [{'end': None,
                          'institution': {'country': 'GB',
                                          'domain': 'lovkush.com',
                                          'name': 'Independent'},
                          'position': 'Researcher',
                          'start': 2024},
                         {'end': 2020,
                          'institution': {'country': 'GB',
                                          'domain': 'le.ac.uk',
                                          'name': 'University of Leicester'},
                          'position': 'Instructor',
                          'start': 2017},
                         {'end': 2016,
                          'institution': {'countr

In [5]:
author_id = '~Lovkush_Agarwal1'
papers = client.get_all_notes(
    content={'authorids': author_id},
    details='replies'
)

In [6]:
print(papers[0])

{'cdate': 1724838331162,
 'content': {'TLDR': {'value': 'We study how language models might encode '
                               'paragraphs, and find newline tokens '
                               'activations do this to some extent.'},
             '_bibtex': {'value': '@inproceedings{\n'
                                  'pochinkov2024extracting,\n'
                                  'title={Extracting Paragraphs from {LLM} '
                                  'Token Activations},\n'
                                  'author={Nicky Pochinkov and Angelo Benoit '
                                  'and Lovkush Agarwal and Zainab Ali Majid '
                                  'and Lucile Ter-Minassian},\n'
                                  'booktitle={🍃 MINT: Foundation Model '
                                  'Interventions},\n'
                                  'year={2024},\n'
                                  'url={https://openreview.net/forum?id=4b675AHcqq}\n'
                   

In [7]:
results = {}
for paper in papers:
    paper_info = {
        'title': paper.content['title']['value'],
        'date_creation': datetime.fromtimestamp(paper.cdate/1000).strftime('%Y-%m-%d'),
        # 'paperid': paper.id,
        'venueid': paper.content['venueid']['value'],
        'authorids': paper.content['authorids']['value'],
    }
    # Check for decision in replies
    decision = None
    for reply in paper.details['replies']:
        try:
            decision = reply['content']['decision']['value']
            break
        except:
            pass
    paper_info['decision'] = decision
    results[paper.id] = paper_info

for result in results:
    print(result)
    print(results[result])
    print()

4b675AHcqq
{'title': 'Extracting Paragraphs from LLM Token Activations', 'date_creation': '2024-08-28', 'venueid': 'NeurIPS.cc/2024/Workshop/MINT', 'authorids': ['~Nicky_Pochinkov1', '~Angelo_Benoit1', '~Lovkush_Agarwal1', '~Zainab_Ali_Majid1', '~Lucile_Ter-Minassian1'], 'decision': 'Accept'}



In [8]:
# based on later investigations, create new hacky function to get decision

def get_decision(paper: openreview.api.client.Note) -> tuple[str|None, str|None]:
    decision = None

    # first try to get decision from replies
    for reply in paper.details['replies']:
        try:
            decision = reply['content']['decision']['value']
            return decision, "From decision field in replies"
        except:
            pass
    
    # if that fails, try to get decision from replies but with recommendation field
    for reply in paper.details['replies']:
        try:
            recommendation = reply['content']['recommendation']['value']
            return recommendation, "From recommendation field in replies"
        except:
            pass
    
    # if that fails, try to see if 'Poster' or 'Oral' or 'Spotlight' is at the end of venue
    if 'venue' in paper.content:
        if paper.content['venue']['value'].lower().endswith('poster'):
            return 'Accept Poster', "From venue field"
        elif paper.content['venue']['value'].lower().endswith('oral'):
            return 'Accept Oral', "From venue field"
        elif paper.content['venue']['value'].lower().endswith('spotlight'):
            return 'Accept Spotlight', "From venue field"
    
    return None, "No decision found"
    

# Get names from Airtable

In [9]:
import os
import dotenv
from pyairtable import Api

dotenv.load_dotenv()
api = Api(os.environ['AIRTABLE_API_KEY'])

base_id = "appZq2f1sM0tW9kH7"
table_id = "tblX2K7X7sFF8Q8Be"
table = api.table(base_id, table_id)
table_data = table.all()


In [10]:
names = []
for row in table_data:
    name = row['fields']['Scholar Name']
    name = name.strip().lower() # Remove leading and trailing whitespace, make lower case
    if name == "":
        continue
    if name not in names:
        names.append(name)

# for name in names:
#     print(name)


# Get author ids from all names

In [11]:
# scholar info dict
# keys are names, value is dictionary with two keys: n_profiles and author_id if n_profiles is 1, otherwise None

client = openreview.api.OpenReviewClient(
    baseurl='https://api2.openreview.net',
)

scholar_info = {}

for name in tqdm(names):
    profiles = client.search_profiles(term=name)
    n_profiles = len(profiles)
    if n_profiles == 1:
        author_id = profiles[0].id
    else:
        author_id = None 
    scholar_info[name] = {
        'n_profiles': n_profiles,
        'author_id': author_id
    }

 64%|██████▍   | 197/306 [00:41<00:21,  4.98it/s]

Retrying request: GET /profiles/search?term=patrick+leask&es=false, response: {'name': 'RateLimitError', 'message': 'Too many requests: You have made 201 requests, surpassing the limit of 200 requests. Please try again in 10 seconds (2025-10-03-3948372)', 'status': 429, 'details': {'limit': 200, 'remaining': 0, 'resetTime': '2025-10-03T14:24:52.549Z', 'used': 201, 'current': 201, 'reqId': '2025-10-03-3948372'}}, error: None


100%|██████████| 306/306 [01:12<00:00,  4.21it/s]


In [12]:
# count how many scholars have 1 profile
n_scholars_with_1_profile = sum(1 for scholar in scholar_info.values() if scholar['n_profiles'] == 1)
print(f"Number of scholars with 1 profile: {n_scholars_with_1_profile}")

# count how many scholars have 0 profiles
n_scholars_with_0_profiles = sum(1 for scholar in scholar_info.values() if scholar['n_profiles'] == 0)
print(f"Number of scholars with 0 profiles: {n_scholars_with_0_profiles}")

# count how many scholars have 2 or more profiles
n_scholars_with_2_or_more_profiles = sum(1 for scholar in scholar_info.values() if scholar['n_profiles'] >= 2)
print(f"Number of scholars with 2 or more profiles: {n_scholars_with_2_or_more_profiles}")

assert n_scholars_with_1_profile + n_scholars_with_0_profiles + n_scholars_with_2_or_more_profiles == len(names)

Number of scholars with 1 profile: 185
Number of scholars with 0 profiles: 75
Number of scholars with 2 or more profiles: 46


In [13]:
# for name, scholar in list(scholar_info.items())[:30]:
#     print(f"{name}, {scholar['n_profiles']}, {scholar['author_id']}")


In [14]:
# get list of author_ids where they are not None
author_ids = [scholar['author_id'] for scholar in scholar_info.values() if scholar['author_id'] is not None]
print(len(author_ids))



185


# get paper stats for all author_ids

In [15]:
all_papers = {}

for author_id in tqdm(author_ids):
    # wait to prevent rate limiting. seems to be limit of 60 requests per minute
    sleep(1)
    
    papers = client.get_all_notes(
        content={'authorids': author_id},
        details='replies'
    )

    for paper in papers:
        if paper.id in all_papers:
            continue
        paper_info = {
            'title': paper.content['title']['value'],
            'date_creation': datetime.fromtimestamp(paper.cdate/1000).strftime('%Y-%m-%d'),
            # 'paperid': paper.id,
            'venueid': paper.content['venueid']['value'],
            'authorids': paper.content['authorids']['value'],
        }
        decision, decision_reasoning = get_decision(paper)
        paper_info['decision'] = decision
        paper_info['decision_reasoning'] = decision_reasoning
        all_papers[paper.id] = paper_info

 19%|█▉        | 35/185 [00:50<03:08,  1.26s/it]

Retrying request: GET /notes?content.authorids=~Thomas_Bush1&limit=1000&details=replies&after=DzGe40glxs&sort=id&count=false, response: {'name': 'RateLimitError', 'message': 'Too many requests: You have made 61 requests, surpassing the limit of 60 requests. Please try again in 10 seconds (2025-10-03-3960127)', 'status': 429, 'details': {'limit': 60, 'remaining': 0, 'resetTime': '2025-10-03T14:26:15.454Z', 'used': 61, 'current': 61, 'reqId': '2025-10-03-3960127'}}, error: None


 37%|███▋      | 68/185 [01:46<02:42,  1.39s/it]

Retrying request: GET /notes?content.authorids=~Tim_Tian_Hua1&limit=1000&details=replies&sort=id&count=true, response: {'name': 'RateLimitError', 'message': 'Too many requests: You have made 61 requests, surpassing the limit of 60 requests. Please try again in 15 seconds (2025-10-03-3965087)', 'status': 429, 'details': {'limit': 60, 'remaining': 0, 'resetTime': '2025-10-03T14:27:15.944Z', 'used': 61, 'current': 61, 'reqId': '2025-10-03-3965087'}}, error: None


 56%|█████▌    | 103/185 [02:48<01:43,  1.26s/it]

...

 76%|███████▌  | 140/185 [03:51<00:59,  1.32s/it]

Retrying request: GET /notes?content.authorids=~Andrew_Mackenzie1&limit=1000&details=replies&after=4VWnC5unAV&sort=id&count=false, response: {'name': 'RateLimitError', 'message': 'Too many requests: You have made 61 requests, surpassing the limit of 60 requests. Please try again in 11 seconds (2025-10-03-3981799)', 'status': 429, 'details': {'limit': 60, 'remaining': 0, 'resetTime': '2025-10-03T14:29:17.982Z', 'used': 61, 'current': 61, 'reqId': '2025-10-03-3981799'}}, error: None


 94%|█████████▍| 174/185 [04:50<00:14,  1.32s/it]

Retrying request: GET /notes?content.authorids=~Oscar_Balcells_Obeso1&limit=1000&details=replies&after=EqF16oDVFf&sort=id&count=false, response: {'name': 'RateLimitError', 'message': 'Too many requests: You have made 61 requests, surpassing the limit of 60 requests. Please try again in 13 seconds (2025-10-03-3987534)', 'status': 429, 'details': {'limit': 60, 'remaining': 0, 'resetTime': '2025-10-03T14:30:18.728Z', 'used': 61, 'current': 61, 'reqId': '2025-10-03-3987534'}}, error: None


100%|██████████| 185/185 [05:18<00:00,  1.72s/it]


In [16]:
# convert dict to dataframe
df = pd.DataFrame(all_papers).T
df.reset_index(inplace=True)
df.rename(columns={'index': 'paperid'}, inplace=True)
df.head()

,paperid,title,date_creation,venueid,authorids,decision,decision_reasoning
0,qzsDKwGJyB,Measuring Progress in Dictionary Learning for ...,2024-05-30,ICML.cc/2024/Workshop/MI,"[~Adam_Karvonen1, ~Benjamin_Wright2, ~Can_Rage...",Accept (Oral),From decision field in replies
1,qrU3yNfX0d,SAEBench: A Comprehensive Benchmark for Sparse...,2025-01-23,ICML.cc/2025/Conference,"[~Adam_Karvonen1, ~Can_Rager1, ~Johnny_Lin1, ~...",Accept (poster),From decision field in replies
2,pZakRK1QHU,Linearly Structured World Representations in M...,2023-10-07,NeurIPS.cc/2023/Workshop/UniReps,"[~Michael_Ivanitskiy1, ~Alexander_F_Spies1, ~T...",Accept (Poster),From decision field in replies
3,bKawydfGhb,An Adversarial Example for Direct Logit Attrib...,2024-05-29,ICML.cc/2024/Workshop/MI,"[~Jett_Janiak1, ~Can_Rager1, ~James_Dao1, ~Yeu...",Accept (Poster),From decision field in replies
4,SCEdoGghcw,Measuring Progress in Dictionary Learning for ...,2024-05-15,NeurIPS.cc/2024/Conference,"[~Adam_Karvonen1, ~Benjamin_Wright2, ~Can_Rage...",Accept (poster),From decision field in replies


In [41]:
df['paper_url'] = df['paperid'].apply(lambda x: f"https://openreview.net/forum?id={x}")
df['decision_simple'] = df['decision'].apply(lambda x: x.split(' ')[0] if x else None).apply(lambda x: x.split('-')[0] if x else None).fillna("UNKNOWN").str.lower()
df.head()

,paperid,title,date_creation,venueid,authorids,decision,decision_reasoning,paper_url,decision_simple
0,qzsDKwGJyB,Measuring Progress in Dictionary Learning for ...,2024-05-30,ICML.cc/2024/Workshop/MI,"[~Adam_Karvonen1, ~Benjamin_Wright2, ~Can_Rage...",Accept (Oral),From decision field in replies,https://openreview.net/forum?id=qzsDKwGJyB,accept
1,qrU3yNfX0d,SAEBench: A Comprehensive Benchmark for Sparse...,2025-01-23,ICML.cc/2025/Conference,"[~Adam_Karvonen1, ~Can_Rager1, ~Johnny_Lin1, ~...",Accept (poster),From decision field in replies,https://openreview.net/forum?id=qrU3yNfX0d,accept
2,pZakRK1QHU,Linearly Structured World Representations in M...,2023-10-07,NeurIPS.cc/2023/Workshop/UniReps,"[~Michael_Ivanitskiy1, ~Alexander_F_Spies1, ~T...",Accept (Poster),From decision field in replies,https://openreview.net/forum?id=pZakRK1QHU,accept
3,bKawydfGhb,An Adversarial Example for Direct Logit Attrib...,2024-05-29,ICML.cc/2024/Workshop/MI,"[~Jett_Janiak1, ~Can_Rager1, ~James_Dao1, ~Yeu...",Accept (Poster),From decision field in replies,https://openreview.net/forum?id=bKawydfGhb,accept
4,SCEdoGghcw,Measuring Progress in Dictionary Learning for ...,2024-05-15,NeurIPS.cc/2024/Conference,"[~Adam_Karvonen1, ~Benjamin_Wright2, ~Can_Rage...",Accept (poster),From decision field in replies,https://openreview.net/forum?id=SCEdoGghcw,accept


In [42]:
df['decision_simple'].value_counts(dropna=False)

decision_simple
accept     322
unknown    152
reject      50
Name: count, dtype: int64

In [43]:
df['decision_reasoning'].value_counts(dropna=False)

decision_reasoning
From decision field in replies          212
No decision found                       152
From venue field                        144
From recommendation field in replies     16
Name: count, dtype: int64

# basic analysis by conference

In [44]:
venue_counts = df['venueid'].value_counts(dropna=False)
venue_counts.head(30)


venueid
NeurIPS.cc/2025/Workshop/MechInterp              38
ICLR.cc/2025/Conference/Rejected_Submission      33
dblp.org/journals/CORR/2024                      33
ICML.cc/2025/Conference                          28
ICLR.cc/2025/Conference                          26
ICML.cc/2024/Workshop/MI                         21
NeurIPS.cc/2024/Conference                       17
TMLR                                             16
ICLR.cc/2024/Conference/Rejected_Submission      13
NeurIPS.cc/2024/Workshop/SoLaR                   12
ICML.cc/2025/Workshop/R2-FM                      10
OpenReview.net/Archive                           10
ICLR.cc/2025/Conference/Withdrawn_Submission     10
dblp.org/journals/CORR/2023                       9
NeurIPS.cc/2024/Workshop/SafeGenAi                8
ICML.cc/2024/Conference                           8
EMNLP/2023/Conference                             7
dblp.org/journals/CORR/2025                       7
ICLR.cc/2024/Conference                           7
ICLR

In [47]:
conferences = [
    'NeurIPS',
    'ICML',
    'ICLR',
    'NeurIPS.cc/2025/Workshop/MechInterp',
    'ICLR.cc/2025/Conference',
    'dblp.org/journals/CORR',
    'ICML.cc/2025/Conference',
    'ICML.cc/2024/Workshop/MI',
    'NeurIPS.cc/2024/Conference',
    'TMLR',
    'NeurIPS.cc/2025/Workshop/LLM_Evaluation',
    'NeurIPS.cc/2024/Workshop/SoLaR',
    'ICML.cc/2025/Workshop',
]

for conference in conferences:
    print(f"Venue id contains the string: {conference}")
    conference_decisions = df.loc[df['venueid'].str.contains(conference), 'decision_simple'].value_counts(dropna=False)

    # loop through and print info
    for decision, count in conference_decisions.items():
        print(f"{decision}: {count}")
    print()

Venue id contains the string: NeurIPS
accept: 123
unknown: 9
reject: 3

Venue id contains the string: ICML
accept: 99
unknown: 8

Venue id contains the string: ICLR
accept: 51
reject: 47
unknown: 33

Venue id contains the string: NeurIPS.cc/2025/Workshop/MechInterp
accept: 38

Venue id contains the string: ICLR.cc/2025/Conference
reject: 33
accept: 26
unknown: 11

Venue id contains the string: dblp.org/journals/CORR
unknown: 54

Venue id contains the string: ICML.cc/2025/Conference
accept: 28

Venue id contains the string: ICML.cc/2024/Workshop/MI
accept: 21

Venue id contains the string: NeurIPS.cc/2024/Conference
accept: 17
reject: 2

Venue id contains the string: TMLR
accept: 16
unknown: 2

Venue id contains the string: NeurIPS.cc/2025/Workshop/LLM_Evaluation
accept: 5

Venue id contains the string: NeurIPS.cc/2024/Workshop/SoLaR
accept: 12

Venue id contains the string: ICML.cc/2025/Workshop
accept: 22
unknown: 3



In [ ]:
df[(df['venueid'].str.contains('ICLR.cc/2025/Conference')) & (df['decision'].isna())]

# we see in this instance, decision None corresponds to withdrawals or desk rejections, via the venueid

,paperid,title,date_creation,venueid,authorids,decision,decision_reasoning,paper_url
34,oycEeFXX74,Shell Games: Control Protocols for Adversarial...,2024-09-28,ICLR.cc/2025/Conference/Withdrawn_Submission,"[~Aryan_Bhatt1, ~Cody_Rushing1, ~Adam_Kaufman1...",None,No decision found,https://openreview.net/forum?id=oycEeFXX74
154,licAR8FPTW,Evaluating Oversight Robustness with Incentivi...,2024-09-28,ICLR.cc/2025/Conference/Withdrawn_Submission,"[~Yoav_Tzfati1, ~McKenna_Fitzgerald1, ~Juan_J_...",None,No decision found,https://openreview.net/forum?id=licAR8FPTW
211,WxqWuG431g,The Geometry of Concepts: Sparse Autoencoder F...,2024-09-27,ICLR.cc/2025/Conference/Withdrawn_Submission,"[~Yuxiao_Li2, ~Eric_J_Michaud1, ~David_D._Baek...",None,No decision found,https://openreview.net/forum?id=WxqWuG431g
221,5IZfo98rqr,Decomposing The Dark Matter of Sparse Autoenco...,2024-09-24,ICLR.cc/2025/Conference/Withdrawn_Submission,"[~Joshua_Engels1, ~Logan_Riggs_Smith1, ~Max_Te...",None,No decision found,https://openreview.net/forum?id=5IZfo98rqr
234,ZaOHSBGOhV,SmartBackdoor: Malicious Language Model Agents...,2024-09-27,ICLR.cc/2025/Conference/Withdrawn_Submission,"[~Heng_Wang10, ~Ruiqi_Zhong1, ~Jiaxin_Wen2, ~J...",None,No decision found,https://openreview.net/forum?id=ZaOHSBGOhV
251,sknUS8X9q0,SAGE: Scalable Ground Truth Evaluations for La...,2024-09-27,ICLR.cc/2025/Conference/Withdrawn_Submission,"[~Constantin_Venhoff1, ~Ani_Calinescu1, ~Phili...",None,No decision found,https://openreview.net/forum?id=sknUS8X9q0
267,U0y32WKeOd,Evaluating Synthetic Activations composed of S...,2024-09-27,ICLR.cc/2025/Conference/Desk_Rejected_Submission,"[~Nora_Petrova1, ~Giorgi_Giglemiani1, ~Chatrik...",None,No decision found,https://openreview.net/forum?id=U0y32WKeOd
271,SLufnMLhbv,GUIDE: Guidance-based Incremental Learning wit...,2024-09-27,ICLR.cc/2025/Conference/Withdrawn_Submission,"[~Bartosz_Cywiński1, ~Kamil_Deja1, ~Tomasz_Trz...",None,No decision found,https://openreview.net/forum?id=SLufnMLhbv
289,XdRv6I80L1,Plan B: Training LLMs to fail less severely,2024-09-24,ICLR.cc/2025/Conference/Withdrawn_Submission,"[~Julian_Stastny1, ~Niels_Warncke1, ~Dylan_Xu2...",None,No decision found,https://openreview.net/forum?id=XdRv6I80L1
314,Ch8s4FdUXS,Unpacking SDXL Turbo: Interpreting Text-to-Ima...,2024-09-27,ICLR.cc/2025/Conference/Withdrawn_Submission,"[~Viacheslav_Surkov1, ~Chris_Wendler1, ~Mikhai...",None,No decision found,https://openreview.net/forum?id=Ch8s4FdUXS


In [34]:
df[(df['venueid'].str.lower().str.contains('withdrawn')) & (df['decision'].isna())]

,paperid,title,date_creation,venueid,authorids,decision,decision_reasoning,paper_url
32,bIb1xhSCVY,Interpreting Reward Models in RLHF-Tuned Langu...,2023-09-22,ICLR.cc/2024/Conference/Withdrawn_Submission,"[~Luke_Marks2, ~Amir_Abdullah1, ~Luna_Mendez1,...",None,No decision found,https://openreview.net/forum?id=bIb1xhSCVY
34,oycEeFXX74,Shell Games: Control Protocols for Adversarial...,2024-09-28,ICLR.cc/2025/Conference/Withdrawn_Submission,"[~Aryan_Bhatt1, ~Cody_Rushing1, ~Adam_Kaufman1...",None,No decision found,https://openreview.net/forum?id=oycEeFXX74
141,gDycDxX5Wa,PatchCraft: Learning Optimized Image Patch for...,2023-09-19,ICLR.cc/2024/Conference/Withdrawn_Submission,"[~Razieh_Rezaei1, ~Masoud_Jalili_Sabet2, ~Ashk...",None,No decision found,https://openreview.net/forum?id=gDycDxX5Wa
154,licAR8FPTW,Evaluating Oversight Robustness with Incentivi...,2024-09-28,ICLR.cc/2025/Conference/Withdrawn_Submission,"[~Yoav_Tzfati1, ~McKenna_Fitzgerald1, ~Juan_J_...",None,No decision found,https://openreview.net/forum?id=licAR8FPTW
211,WxqWuG431g,The Geometry of Concepts: Sparse Autoencoder F...,2024-09-27,ICLR.cc/2025/Conference/Withdrawn_Submission,"[~Yuxiao_Li2, ~Eric_J_Michaud1, ~David_D._Baek...",None,No decision found,https://openreview.net/forum?id=WxqWuG431g
221,5IZfo98rqr,Decomposing The Dark Matter of Sparse Autoenco...,2024-09-24,ICLR.cc/2025/Conference/Withdrawn_Submission,"[~Joshua_Engels1, ~Logan_Riggs_Smith1, ~Max_Te...",None,No decision found,https://openreview.net/forum?id=5IZfo98rqr
234,ZaOHSBGOhV,SmartBackdoor: Malicious Language Model Agents...,2024-09-27,ICLR.cc/2025/Conference/Withdrawn_Submission,"[~Heng_Wang10, ~Ruiqi_Zhong1, ~Jiaxin_Wen2, ~J...",None,No decision found,https://openreview.net/forum?id=ZaOHSBGOhV
242,sNWQUTkDmA,Observable Propagation: Uncovering Feature Vec...,2023-09-22,ICLR.cc/2024/Conference/Withdrawn_Submission,"[~Jacob_Dunefsky1, ~Arman_Cohan1]",None,No decision found,https://openreview.net/forum?id=sNWQUTkDmA
251,sknUS8X9q0,SAGE: Scalable Ground Truth Evaluations for La...,2024-09-27,ICLR.cc/2025/Conference/Withdrawn_Submission,"[~Constantin_Venhoff1, ~Ani_Calinescu1, ~Phili...",None,No decision found,https://openreview.net/forum?id=sknUS8X9q0
271,SLufnMLhbv,GUIDE: Guidance-based Incremental Learning wit...,2024-09-27,ICLR.cc/2025/Conference/Withdrawn_Submission,"[~Bartosz_Cywiński1, ~Kamil_Deja1, ~Tomasz_Trz...",None,No decision found,https://openreview.net/forum?id=SLufnMLhbv


In [35]:
df[(df['venueid'].str.lower().str.contains('rejected')) & (df['decision'].isna())]

,paperid,title,date_creation,venueid,authorids,decision,decision_reasoning,paper_url
267,U0y32WKeOd,Evaluating Synthetic Activations composed of S...,2024-09-27,ICLR.cc/2025/Conference/Desk_Rejected_Submission,"[~Nora_Petrova1, ~Giorgi_Giglemiani1, ~Chatrik...",None,No decision found,https://openreview.net/forum?id=U0y32WKeOd


In [24]:
df[(df['venueid'].str.contains('dblp.org/journals/CORR/2024')) & (df['decision'].isna())].head(5)

# doing some searching seems to show that dblp is just some subset of arxiv?? and openreview for some reason indexes this...? does not actually seem to be any kind of review or conference.

,paperid,title,date_creation,venueid,authorids,decision,decision_reasoning,paper_url
18,8Pf40Qhbq5,Separating Tongue from Thought: Activation Pat...,2024-01-01,dblp.org/journals/CORR/2024,"[~Clément_Dumas1, ~Chris_Wendler1, https://dbl...",None,No decision found,https://openreview.net/forum?id=8Pf40Qhbq5
27,a0Dw4niLjh,Obfuscated Activations Bypass LLM Latent-Space...,2024-01-01,dblp.org/journals/CORR/2024,"[~Luke_Bailey1, ~Alex_Serrano1, https://dblp.o...",None,No decision found,https://openreview.net/forum?id=a0Dw4niLjh
28,JJhPn1AgUi,xCOMET-lite: Bridging the Gap Between Efficien...,2024-01-01,dblp.org/journals/CORR/2024,[https://dblp.org/search/pid/api?q=author:Dani...,None,No decision found,https://openreview.net/forum?id=JJhPn1AgUi
30,9u9Zws3wGP,ViSTa Dataset: Do vision-language models under...,2024-01-01,dblp.org/journals/CORR/2024,[https://dblp.org/search/pid/api?q=author:Evze...,None,No decision found,https://openreview.net/forum?id=9u9Zws3wGP
121,iktWqWaheH,Limitations of Agents Simulated by Predictive ...,2024-01-01,dblp.org/journals/CORR/2024,[https://dblp.org/search/pid/api?q=author:Raym...,None,No decision found,https://openreview.net/forum?id=iktWqWaheH


In [25]:
df[(df['venueid'].str.contains('TMLR')) & (df['decision'].isna())].head(5)

,paperid,title,date_creation,venueid,authorids,decision,decision_reasoning,paper_url
407,KQOHBsXe6b,Beyond the Imitation Game: Quantifying and ext...,2023-01-01,dblp.org/journals/TMLR/2023,[https://dblp.org/search/pid/api?q=author:Aaro...,None,No decision found,https://openreview.net/forum?id=KQOHBsXe6b
453,lPOQI7Kcb3,Inverse Scaling: When Bigger Isn't Better,2023-01-01,dblp.org/journals/TMLR/2023,"[~Ian_R._McKenzie1, ~Alexander_Lyzhov1, https:...",None,No decision found,https://openreview.net/forum?id=lPOQI7Kcb3


In [29]:
df[(df['venueid'].str.contains('ICML.cc/2025/Workshop')) & (df['decision'].isna())].head(5)

,paperid,title,date_creation,venueid,authorids,decision,decision_reasoning,paper_url
170,aMaSHy8IgK,Probing the Limits of Mathematical World Model...,2025-05-20,ICML.cc/2025/Workshop/World_Models,"[~Henry_Kvinge1, ~Elizabeth_Coda1, ~Eric_Yeats...",None,No decision found,https://openreview.net/forum?id=aMaSHy8IgK
171,VbqocgftJF,Permutations as a testbed for studying the eff...,2025-05-26,ICML.cc/2025/Workshop/MOSS,"[~Sarah_McGuire_Scullen1, ~Davis_Brown1, ~Robe...",None,No decision found,https://openreview.net/forum?id=VbqocgftJF
384,77w4a4oxpq,Evaluating Forecasting is More Difficult than ...,2025-05-22,ICML.cc/2025/Workshop/World_Models,"[~Daniel_Paleka1, ~Shashwat_Goel1, ~Jonas_Geip...",None,No decision found,https://openreview.net/forum?id=77w4a4oxpq


In [31]:
# can maybe use the invitations field to get information??
# 'invitations': ['ICML.cc/2025/Workshop/World_Models/-/Submission',
                #  'ICML.cc/2025/Workshop/World_Models/-/Post_Submission',
                #  'ICML.cc/2025/Workshop/World_Models/-/Edit',
                #  'ICML.cc/2025/Workshop/World_Models/Submission7/-/Camera-Ready'],
paper_id = "aMaSHy8IgK"
paper = client.get_all_notes(id=paper_id, details='replies')
print(paper[0])


{'cdate': 1747698466524,
 'content': {'TLDR': {'value': 'We investigate whether the mathematical world '
                               'models of LLMs align with structures and '
                               'properties from mathematics broadly'},
             '_bibtex': {'value': '@inproceedings{\n'
                                  'kvinge2025probing,\n'
                                  'title={Probing the Limits of Mathematical '
                                  'World Models in {LLM}s},\n'
                                  'author={Henry Kvinge and Elizabeth Coda and '
                                  'Eric Yeats and Davis Brown and John '
                                  'Buckheit and Sarah McGuire Scullen and '
                                  'Brendan Kennedy and Loc Truong and William '
                                  'Kay and Cliff Joslyn and Tegan Emerson and '
                                  'Michael J. Henry and John Anthony '
                                  '